In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np

## Import data 

In [2]:
dta_dir = 'S:/LLC_0028/data/'

In [3]:
dta =  pd.read_csv(dta_dir + './harmonised_all/llc_0028_full_harmonised_data_all_v2.csv', index_col=0)

## Select columns

In [4]:
core_symp = ['fever','cough','throat',
            'chest_tight','breath','nose','aches',
             'fatigue', 'diarrhoea','smell_taste',
            'nausea_vomit','sneezing',
             'headache','concentrating','memory']

symp = dta.loc[~(dta[core_symp].isna().any(axis=1))]['LLC_0028_stud_id'].values

In [5]:
dta = dta.loc[dta['LLC_0028_stud_id'].isin(symp)]

In [7]:
sociodemo = ['llc_age','llc_ethnic3','llc_ethnic7','llc_sex','functional_limitation_cat']

In [8]:
dta = dta[['LLC_0028_stud_id','study', 'covid_status'] + sociodemo ]

## Bin age 

In [9]:
age_bins = pd.cut(dta.llc_age, bins = [18,45,55,65,100], labels = ['18-45','45-55','55-65','65+'])

In [10]:
dta['age_cat'] = age_bins

In [12]:
#need to encode as numerics for stata.
#map using dict

agedict = {'18-45':1,
           '45-55':0,#reference
           '55-65':2,
           '65+':3} #nan will be encoded as 99

dta['age_cat_numeric'] = [agedict[v] if v in agedict else 99 
                          for v in dta.age_cat.values]

In [13]:
dta.age_cat_numeric.unique()

array([ 0, 99,  1,  2,  3], dtype=int64)

## Encode missing ethnicity 

In [14]:
dta.llc_ethnic3.unique()

array([ 0., nan,  7., 99.])

In [15]:
# 99 mans missing in llc_ethnic3 but some values are nan --- recoed all missing values as 99
dta['llc_ethnic3'] = [int(v) if v in [7.,0.] else 99 for i,v in enumerate(dta.llc_ethnic3.values)]

## Encode FL 

In [16]:
# -1 means data not collected -- need to exclude these rows in mlr
# no encoding needed
dta.functional_limitation_cat.unique()

array([ 0,  2, -1,  1,  4,  3], dtype=int64)

## Link to LCA clusters 

### No covid 

In [17]:
lca_file = pd.read_csv(dta_dir + 'lca_results/v2/core_pooled_nocovid_2_classes.csv', index_col=0)

##### Replace LLC_0028_stud_id in lca_file 

ids = dta.loc[(dta.covid_status==0)]['LLC_0028_stud_id']

print(ids.shape[0] == lca_file.shape[0])

lca_file['LLC_0028_stud_id'] = ids.values
lca_file['lca_cluster'] = lca_file.lc4.values

merged = dta.merge(lca_file[['LLC_0028_stud_id','lca_cluster']], left_on = 'LLC_0028_stud_id',
                      right_on = 'LLC_0028_stud_id',
                      how='left')

merged = merged.loc[merged.covid_status==0]

largest = merged.lca_cluster.value_counts().nlargest(1).index[0]

merged['lca_cluster'] = [v if v!=largest else -10 for v in merged.lca_cluster.values]

True


In [19]:
merged.to_csv(dta_dir + './cluster_comparison/'+'no_covid_mlr_complete_dta.csv')
no = merged

### Covid < 12 weeks ago 

In [35]:
lca_file = pd.read_csv(dta_dir + 'lca_results/v2/core_pooled_l12_2_classes.csv', index_col=0)


##### Replace LLC_0028_stud_id in lca_file 

ids = dta.loc[(dta.covid_status==1)]['LLC_0028_stud_id']

print(ids.shape[0] == lca_file.shape[0])

lca_file['LLC_0028_stud_id'] = ids.values
lca_file['lca_cluster'] = lca_file.lc4.values

merged = dta.merge(lca_file[['LLC_0028_stud_id','lca_cluster']], left_on = 'LLC_0028_stud_id',
                      right_on = 'LLC_0028_stud_id',
                      how='left')

merged = merged.loc[merged.covid_status==1]

largest = merged.lca_cluster.value_counts().nlargest(1).index[0]

merged['lca_cluster'] = [v if v!=largest else -10 for v in merged.lca_cluster.values]

True


In [37]:
merged.to_csv(dta_dir + './cluster_comparison/'+'l12_mlr_complete_dta.csv')

### Covid > 12 weeks ago

In [32]:
lca_file = pd.read_csv(dta_dir + 'lca_results/v2/core_pooled_g12_2_classes.csv', index_col=0)

##### Replace LLC_0028_stud_id in lca_file 

ids = dta.loc[(dta.covid_status==2)]['LLC_0028_stud_id']

print(ids.shape[0] == lca_file.shape[0])

lca_file['LLC_0028_stud_id'] = ids.values
lca_file['lca_cluster'] = lca_file.lc4.values

merged = dta.merge(lca_file[['LLC_0028_stud_id','lca_cluster']], left_on = 'LLC_0028_stud_id',
                      right_on = 'LLC_0028_stud_id',
                      how='left')

merged = merged.loc[merged.covid_status==2]

largest = merged.lca_cluster.value_counts().nlargest(1).index[0]

merged['lca_cluster'] = [v if v!=largest else -10 for v in merged.lca_cluster.values]

True


In [33]:
merged.to_csv(dta_dir + './cluster_comparison/'+'g12_mlr_complete_dta.csv')

### All categories 

In [25]:
lca_file = pd.read_csv(dta_dir + 'lca_results/v2/core_pooled_all_2_classes.csv', index_col=0)

##### Replace LLC_0028_stud_id in lca_file 

ids = dta['LLC_0028_stud_id']

print(ids.shape[0] == lca_file.shape[0])

lca_file['LLC_0028_stud_id'] = ids.values
lca_file['lca_cluster'] = lca_file.lc4.values

merged = dta.merge(lca_file[['LLC_0028_stud_id','lca_cluster']], left_on = 'LLC_0028_stud_id',
                      right_on = 'LLC_0028_stud_id',
                      how='left')

#merged = merged.loc[merged.covid_status==1]

largest = merged.lca_cluster.value_counts().nlargest(1).index[0]

merged['lca_cluster'] = [v if v!=largest else -10 for v in merged.lca_cluster.values]

True


In [26]:
merged.shape

(26144, 11)

In [27]:
merged.to_csv(dta_dir + './cluster_comparison/'+'all_mlr_complete_dta.csv')